## 0. Setup (once per environment)

If you're running on a fresh environment, install the packages below.  
If you're on a managed environment (e.g., classroom PCs), you can skip the install and just run the imports.

In [ ]:
# If needed, uncomment the next line (and re-run) to install/upgrade Qiskit locally.
!pip -q install "qiskit>=1.1" "qiskit-aer>=0.14" "qiskit-ibm-runtime>=0.25" --upgrade
!pip install -q pylatexenc

In [ ]:
from qiskit import QuantumCircuit, transpile, QuantumRegister, ClassicalRegister
from qiskit.visualization import plot_histogram,\
                                 plot_bloch_multivector,\
                                 plot_distribution
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
# Import the Parameter module from the Qiskit's Circuit module
from qiskit.circuit import Parameter
# Import the Options, Session, and Sampler v2 modules
# from the Qiskit's IBM Runtime library
from qiskit_ibm_runtime import Options, Session,\
                               SamplerV2 as Sampler
# Import the Generate Preset Pass Manager module
# from the Qiskit's Transpiler module
from qiskit.transpiler import generate_preset_pass_manager

from qiskit.result import marginal_counts



# (Optional) Fake backends with realistic noise (no IBM account required)
try:
    from qiskit_ibm_runtime.fake_provider import FakeManilaV2
    HAS_FAKE = True
except Exception:
    HAS_FAKE = False

from math import pi
import numpy as np
import matplotlib.pyplot as plt

# Reproducibility for simulators that support it
np.random.seed(42)
print("Imports OK. Qiskit notebook is ready!")
print("Fake noisy backend available:", HAS_FAKE)

### Helper Utilities

We’ll use these helpers to simulate circuits and check your answers.

In [ ]:
def run_counts(qc, shots=1000, seed=42):
    """Compile and run a circuit on AerSimulator and return counts."""
    sim = AerSimulator(seed_simulator=seed)
    tqc = transpile(qc, sim)
    res = sim.run(tqc, shots=shots).result()
    return res.get_counts()


def noise_run_counts(qc, shots=1000, seed=42):
    """Compile and run a circuit on AerSimulator and return counts."""
    sim = FakeManilaV2()
    tqc = transpile(qc, sim)
    res = sim.run(tqc, shots=shots).result()
    return res.get_counts()


def probs_from_counts(counts):
    total = sum(counts.values())
    return {k: v/total for k, v in counts.items()}

def almost_equal(a, b, tol=0.05):
    return abs(a - b) <= tol

def complete_counts(counts: dict) -> dict:
    for key in counts:
      n_bits = len(key)
      break
    all_keys = [format(i, f'0{n_bits}b') for i in range(2**n_bits)]
    return {k: counts.get(k, 0) for k in all_keys}



## 1. Multi-Qubit Gates and Entanglement

### 1.1 Controlled-NOT (CNOT) Gate

Having demonstrated quantum superposition, we now turn to another fundamental quantum gate—the **Controlled-NOT (CNOT) gate**—which plays a crucial role in creating **entanglement** between two qubits. Unlike single-qubit gates, the CNOT gate operates on **two qubits**: a **control qubit** and a **target qubit**. Its action is conditional: it applies a NOT operation (i.e., flips the target qubit) **only if** the control qubit is in the state $|1\rangle$.





The CNOT gate is represented by a **4×4 unitary matrix**, since it acts on the combined state space of two qubits:

\begin{align}
\text{CNOT} = \begin{pmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 0 & 1 \\
0 & 0 & 1 & 0
\end{pmatrix}
\end{align}


This matrix maps the four computational basis states as follows:

$$\text{CNOT}|00\rangle = |00\rangle$$  
$$\text{CNOT}|01\rangle = |01\rangle$$  
$$\text{CNOT}|10\rangle = |11\rangle$$  
$$\text{CNOT}|11\rangle = |10\rangle$$  

As shown, the target qubit is flipped **only** when the control qubit is $|1\rangle$. This gate is essential for generating entangled states such as the Bell states, which are foundational in quantum communication and quantum error correction.

- Key Take-away: the **CNOT** gate flips the target qubit only if the control is $|1\rangle$.





In [ ]:
# Create the two-qubit quantum circuit
qc = QuantumCircuit(2)

# Apply an Hadamard gate to qubit 0
qc.cx(0,1)

# Draw the circuit
qc.draw(output="mpl")

In [ ]:
# See the statevector
out_vector = Statevector(qc)

print("\n", out_vector, "\n")

# Draw a Bloch sphere
plot_bloch_multivector(out_vector)

Now let's apply the X gate to flip the control bit before applying CNOT. Pay attention to what will happen to the target qubit

In [ ]:
# Create the two-qubit quantum circuit
qc = QuantumCircuit(2)
# Apply X gate to qubit 0
qc.x(0)


# Draw the circuit
qc.draw(output="mpl")

In [ ]:
# See the statevector
out_vector = Statevector(qc)

print("\n", out_vector, "\n")

# Draw a Bloch sphere
print("\n" ,"Qubits states before Applying the CONT gate" , "\n")
plot_bloch_multivector(out_vector)

In [ ]:
# Apply a CNOT gate: control qubit = qubit 0, target qubit = qubit 1
qc.cx(0,1)
# See the statevector
out_vector = Statevector(qc)
print("\n", out_vector, "\n")

# Draw a Bloch sphere
print("\n" ,"Qubits states after Applying the CONT gate" , "\n")
plot_bloch_multivector(out_vector)

### 1.2 Entanglement and Bell States

Now we have all the gates required to entangle 2 qubit.
**Entanglement** is a uniquely quantum phenomenon in which the states of two or more qubits become correlated in such a way that the state of one qubit cannot be described independently of the state of the other—even when separated by large distances. This non-classical correlation is a cornerstone of quantum information science, enabling protocols such as quantum teleportation, superdense coding, and quantum cryptography.



The most well-known examples of entangled states are the **Bell states**, which form a complete orthonormal basis for the two-qubit Hilbert space. These states are defined as:


\begin{aligned}
|\Phi^+\rangle &= \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle) \\
|\Phi^-\rangle &= \frac{1}{\sqrt{2}}(|00\rangle - |11\rangle) \\
|\Psi^+\rangle &= \frac{1}{\sqrt{2}}(|01\rangle + |10\rangle) \\
|\Psi^-\rangle &= \frac{1}{\sqrt{2}}(|01\rangle - |10\rangle)
\end{aligned}


Each Bell state exhibits **maximal entanglement**, meaning that measurement of one qubit instantaneously determines the state of the other.




### Generating a Bell State from $|00\rangle$

To create an entangled Bell state, we can apply a **Hadamard gate** to the first qubit (the control), followed by a **CNOT gate** acting on both qubits. Starting from the initial state $|00\rangle$:

\begin{align}
   H|0\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)
\end{align}
   So the combined state becomes:
\begin{align}
   \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle) \otimes |0\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |10\rangle)
\end{align}

**Apply CNOT with the first qubit as control and the second as target**:
\begin{align}
   \text{CNOT}\left(\frac{1}{\sqrt{2}}(|00\rangle + |10\rangle)\right) = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)
\end{align}

This final state is the **Bell state** $|\Phi^+\rangle$, demonstrating entanglement between the two qubits.


- **NB**:  **Entanglement** is a non-classical correlation where qubit states cannot be described independently.

🧩 Challenge (10mins)
#### Generating the Bell state
$|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|
00\rangle + |11\rangle)$

In [ ]:
# Create the two-qubit quantum circuit
qc = QuantumCircuit(2,2)


# TODO: Put your code here




# Your code ends here

# See the statevector
out_vector = Statevector(qc)
out_vector.draw("latex")



In [ ]:
# Draw the circuit
qc.draw(output="mpl")

We can now execute the quantum circuit to verify whether the resulting state exhibits a 50% probability of measuring either $|00\rangle$ or $|11\rangle$, as expected from the entangled Bell state $|\Phi^+\rangle$.

In [ ]:
# Measure the qubits
qc.measure([0,1], [0,1])

# Get the counts (measurement results)
c = run_counts(qc, shots=4000)

# Run with noise
# c = noise_run_counts(qc, shots=4000)

# Get the measurement probabilities
probs = probs_from_counts(complete_counts(c))

plot_histogram(probs, title="Your Bell variant (Z-basis)")

 🧩 Challenge 4 — Build a different Bell state (20mins)

Construct one of $|\Phi^-\rangle, |\Psi^+\rangle, |\Psi^-\rangle$ and sample it.  
**Hint:** Keep the final state always in front of you and think what gates you need to apply to get there. We belive you got this.

In [ ]:
# YOUR CODE HERE
qc_mybell = QuantumCircuit(2,2)


# TODO: create a Bell variant, then measure both




# Your code ends here

# Draw the circuit
qc_mybell.draw('mpl')

In [ ]:
out_vector = Statevector(qc_mybell)

# Print the minus state in Dirac notation
out_vector.draw('latex')

In [ ]:
# Measure the qubits
qc_mybell.measure([0,1], [0,1])

# Get the counts (measurement results)
c = run_counts(qc_mybell, shots=4000)

probs = probs_from_counts(complete_counts(c))

plot_histogram(probs, title="Your Bell variant (Z-basis)")


---

In [ ]:
# YOUR CODE HERE
qc_mybell = QuantumCircuit(2,2)


# TODO: create a Bell variant, then measure both



# Your code ends here


# Draw the circuit
qc_mybell.draw('mpl')

In [ ]:

out_vector = Statevector(qc_mybell)

# Print the minus state in Dirac notation
out_vector.draw('latex')

In [ ]:
# Measure the qubits
qc_mybell.measure([0,1], [0,1])

# Get the counts (measurement results)
c = run_counts(qc_mybell, shots=4000)

probs = probs_from_counts(complete_counts(c))

plot_histogram(probs, title="Your Bell variant (Z-basis)")


In [ ]:
# YOUR CODE HERE
qc_mybell = QuantumCircuit(2,2)


# TODO: create a Bell variant, then measure both


# Your code ends here

# Draw the circuit
qc_mybell.draw('mpl')

In [ ]:
out_vector = Statevector(qc_mybell)

# Print the minus state in Dirac notation
out_vector.draw('latex')

In [ ]:
# Measure the qubits
qc_mybell.measure([0,1], [0,1])

# Get the counts (measurement results)
c = run_counts(qc_mybell, shots=4000)

probs = probs_from_counts(complete_counts(c))

plot_histogram(probs, title="Your Bell variant (Z-basis)")


# Quantum Teleportation — Comprehensive Walkthrough

This section gives a **conceptual** and **mathematical** explanation of the teleportation protocol and provides **executable code** that verifies each step.

---

## 1) Ingredients & roles

- **Qubits**: three in total.  
  - `q0` (Alice's message) initially holds the unknown state $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$.  
  - `q1` (Alice's half of an EPR pair).  
  - `q2` (Bob's half of the EPR pair).  
- **Shared resource**: the Bell state $|B_{00}\rangle=|\Phi^+\rangle_{12} = \frac{|00\rangle+|11\rangle}{\sqrt 2}$ across `q1,q2`.
- **Classical communication**: two classical bits from Alice to Bob conveying her measurement outcomes.
- **Goal**: end with `q2` in the original state $|\psi\rangle$, while Alice's qubits are measured (destroying her copy) — consistent with the **no‑cloning theorem**.


teleportation.svg



---

## 2) Algebraic derivation (why it works)

Start in
$$|\Psi_0\rangle = |\psi\rangle_0 \otimes |\Phi^+\rangle_{12}
= (\alpha|0\rangle_0 + \beta|1\rangle_0)\otimes \frac{|00\rangle_{12}+|11\rangle_{12}}{\sqrt2}.$$

Alice performs a **CNOT** from `q0` to `q1`, then a **Hadamard** on `q0`. After these two gates, the joint state can be rearranged as

$$
|\Psi_2\rangle = \frac{1}{2}\Big(
|00\rangle_{01}(\alpha|0\rangle_2+\beta|1\rangle_2) +
|01\rangle_{01}(\alpha|1\rangle_2+\beta|0\rangle_2) + \\
|10\rangle_{01}(\alpha|0\rangle_2-\beta|1\rangle_2) +
|11\rangle_{01}(\alpha|1\rangle_2-\beta|0\rangle_2)
\Big).
$$

Thus Alice's **two‑bit** measurement $(m_0 m_1)$ on $(q0,q1)$ collapses Bob to one of four states, each related to $|\psi\rangle$ by a simple Pauli:

| Alice outcome $(m_0\,m_1)$ | Bob's state on `q2` | Bob's correction |
|---|---|---|
| 00 | $|\psi\rangle$ | $I$ |
| 01 | $X|\psi\rangle$ | apply $X$ |
| 10 | $Z|\psi\rangle$ | apply $Z$ |
| 11 | $ZX|\psi\rangle$ | apply $X$ then $Z$ |

After Bob applies the indicated **Pauli correction**, `q2` is exactly in $|\psi\rangle$. Alice's copy is gone because she **measures** her qubits.

> **Key intuition:** $|\Phi^+\rangle$ stores perfect **correlations**. Alice's Bell‑basis measurement steers Bob's qubit into a state that is *Pauli‑twisted* relative to $|\psi\rangle$. Two classical bits tell Bob **which** twist to undo.

---




## 3) Challenge -- Build quantum teleportation circuit (40mins)

You code will build a teleportation circuit that actually **measures** and then uses `if_test` control‑flow to apply the right corrections to Bob.



In [ ]:
# Two helpful functions
# function that takes the counts dict and return the percentage of each state with respect to the number of shots
def to_percent(counts):
    d = dict(counts)
    tot = sum(d.values()) or 1
    return {k: 100*v/tot for k,v in d.items()}

def prepare_state(qc, theta=1.0, phi=0.3, qubit=0):
    """Prepare |psi> = cos(theta/2)|0> + e^{i phi} sin(theta/2)|1> on 'qubit'."""
    qc.ry(theta, qubit); qc.rz(phi, qubit)

**Using `if_test` for classical-conditioned gates**

`if_test` lets you **execute gates only when a classical bit (or register) has a given value.**

In [ ]:


q = QuantumRegister(2, "q")
m = ClassicalRegister(2, "m")
qc = QuantumCircuit(q, m)

qc.x(0)
qc.measure(0, m[0])     # store outcome in m[0]
qc.barrier()

# Conditionally flip q1 if m[0] == 1
with qc.if_test((m[0], 1)):
    qc.x(1)


qc.barrier()
qc.measure(q, m)     # measure again to see the effect

qc.draw('mpl')

In [ ]:
counts = run_counts(qc, shots=2000)
probs = probs_from_counts(complete_counts(counts))
print(probs)

In [ ]:

"""Build teleportation with control-flow corrections and Bob measured in basis ∈ {Z,X,Y}."""

# Registers
q = QuantumRegister(3, 'q')           # qubits
m0 = ClassicalRegister(1, 'm0_alice')       # Alice result of q0
m1 = ClassicalRegister(1, 'm1_alice')       # Alice result of q1
out = ClassicalRegister(1, 'Bob')     # Bob's final readout
qc = QuantumCircuit(q, m0, m1, out)

# 1) Prepare unknown |psi> on q0
theta, phi = 1.1, 0.4   # these values can be changed
# prepare Alice state that you want to teleporate using the func. prepare_state



qc.barrier()


# 2) Share Bell pair between q1 and q2




qc.barrier()






qc.draw("mpl")





In [ ]:
# 3) Alice's Bell measurement on (q0,q1)



qc.barrier()

# 4) Bob's conditional corrections



qc.barrier()



# 4) Bob's measurement on (q0,q1)




# Classical bit index for 'out[0]' (needed for marginalization)
out_index = qc.find_bit(out[0]).index


#
qc.draw('mpl')

### 3.1 Quick verification on standard states

In [ ]:
# We wrap everything we've done so far inside a function
def teleport_circuit(theta=1.0, phi=0.3, measure_basis='Z'):
    """Build teleportation with control-flow corrections and Bob measured in basis ∈ {Z,X,Y}."""
    q = QuantumRegister(3, 'q')
    m0 = ClassicalRegister(1, 'm0')  # result of q0
    m1 = ClassicalRegister(1, 'm1')  # result of q1
    out = ClassicalRegister(1, 'out')# Bob's final readout
    qc = QuantumCircuit(q, m0, m1, out)



In [ ]:
tests = [
    ("|0>", 0.0, 0.0, {'Z':'0'}),
    ("|1>", np.pi, 0.0, {'Z':'1'}),
    ("|+>", np.pi/2, 0.0, {'X':'0'}),
    ("|->", np.pi/2, np.pi, {'X':'1'}),
    ("|+i>", np.pi/2, np.pi/2, {'Y':'0'}),
]

for name, th, ph, expect in tests:
    for basis, wanted in expect.items():
        qc, out_idx = teleport_circuit(theta=th, phi=ph, measure_basis=basis)
        c = run_counts(qc, shots=2000)                  # full counts (includes Alice’s bits)
        cm = marginal_counts(c, indices=[out_idx])      # just Bob’s bit
        d = dict(cm)
        tot = sum(d.values()) or 1
        pct = {k: f"{100*v/tot:.1f}%" for k, v in d.items()}
        print(f"{name} in {basis} → {pct}  (expect {wanted})")


## 4) What happens **without** the corrections?

Run the same circuit but **omit** Bob's Pauli fixes; now his outcome depends on Alice's two classical bits. This is where the two-bit message matters.


In [ ]:
def branch_without_corrections(theta=1.0, phi=0.3, measure_basis='Z'):
    q = QuantumRegister(3, 'q')
    m0 = ClassicalRegister(1, 'm0')  # result of q0
    m1 = ClassicalRegister(1, 'm1')  # result of q1
    out = ClassicalRegister(1, 'out')# Bob's final readout
    qc = QuantumCircuit(q, m0, m1, out)

    # 1) Prepare unknown |psi> on q0
    prepare_state(qc, theta, phi, 0)

    # 2) Share Bell pair between q1 and q2
    qc.h(1); qc.cx(1,2)

    # 3) Alice's Bell measurement on (q0,q1)
    qc.cx(0,1); qc.h(0)
    qc.measure(0, m0[0])
    qc.measure(1, m1[0])

    # # 4) Bob's conditional corrections
    # with qc.if_test((m0[0], 1)):
    #     qc.z(2)
    # with qc.if_test((m1[0], 1)):
    #     qc.x(2)

    # 5) Choose measurement basis for Bob
    if measure_basis == 'X':
        qc.h(2)
    elif measure_basis == 'Y':
        qc.sdg(2); qc.h(2)

    qc.measure(2, out[0])

    # Classical bit index for 'out[0]' (needed for marginalization)
    out_index = qc.find_bit(out[0]).index
    return qc, out_index

theta, phi = 0.0, 0.0
qcZ, idxZ = branch_without_corrections(theta, phi, 'Z')
qcX, idxX = branch_without_corrections(theta, phi, 'X')

cz = marginal_counts(run_counts(qcZ), indices=[idxZ])
cx = marginal_counts(run_counts(qcX), indices=[idxX])

print("Bob WITHOUT corrections (Z basis):", dict(cz))
print("Bob WITHOUT corrections (X basis):", dict(cx))


---

## 5) Why this doesn't violate relativity or cloning

- **No faster‑than‑light signalling**: Bob doesn't know which correction to apply until classical bits arrive. The correlations alone carry no usable message.\n
- **No‑cloning**: Alice's qubits are **measured**; her copy is destroyed. Teleportation transfers the **state**, not the particle.\n
- **Resource accounting**: one shared Bell pair + two classical bits per qubit teleported.




Congratulations! You finished this Qiskit Fall Fest 2025 lab! 🎉🎉🎉

Made with ❤ by **Otmane Ainelkitane** (UC OpenSI) & **Pranay Khosla** (ANU).
